## inport

In [1]:
import os
import cv2
import json


## ディレクトリ設定

In [ ]:

# 設定
label_dir = "yolo_labels"            # YOLO .txt ファイルのディレクトリ
image_dir = "images"                 # 対応する画像のディレクトリ
output_dir = "yolo_preview"         # 出力先ディレクトリ
class_map_path = "DATASETTING.json" # クラスIDと名前のマッピング

os.makedirs(output_dir, exist_ok=True)

In [ ]:
# クラスID → クラス名の読み込み
with open(class_map_path, 'r') as f:
    name_to_id = json.load(f)
id_to_name = {v: k for k, v in name_to_id.items()}

# 全てのラベルファイルを処理
for label_file in os.listdir(label_dir):
    if not label_file.endswith(".txt"):
        continue

    # 画像ファイル名推測
    base = os.path.splitext(label_file)[0]
    img_path = None
    for ext in [".jpg", ".png", ".jpeg"]:
        temp_path = os.path.join(image_dir, base + ext)
        if os.path.exists(temp_path):
            img_path = temp_path
            break
    if img_path is None:
        print(f"対応する画像が見つかりません: {base}")
        continue

    # 画像読み込み
    img = cv2.imread(img_path)
    if img is None:
        print(f"画像読み込み失敗: {img_path}")
        continue
    h_img, w_img = img.shape[:2]

    # アノテーション読み込み
    with open(os.path.join(label_dir, label_file), 'r') as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        class_id, cx, cy, w, h = map(float, parts)
        class_id = int(class_id)

        # 元の画像サイズに変換
        x1 = int((cx - w / 2) * w_img)
        y1 = int((cy - h / 2) * h_img)
        x2 = int((cx + w / 2) * w_img)
        y2 = int((cy + h / 2) * h_img)

        # 描画
        class_name = id_to_name.get(class_id, str(class_id))
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img, class_name, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 0, 0), 2, cv2.LINE_AA)

    # 保存
    out_path = os.path.join(output_dir, os.path.basename(img_path))
    cv2.imwrite(out_path, img)

    # 表示（必要な場合）
    # cv2.imshow("YOLO Preview", img)
    # cv2.waitKey(0)

# cv2.destroyAllWindows()